In [ ]:
from peft import PromptTuningConfig, TaskType
from scripts import train_peft

peft_config = PromptTuningConfig(
    task_type=TaskType.SEQ_CLS,
    num_virtual_tokens=4,
)
train_peft(peft_config)

In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoProcessor, AutoModelForCausalLM, DataCollatorForSeq2Seq,TrainingArguments,Trainer

/home/chathurya/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds = load_dataset("arampacha/rsicd")
ds

DatasetDict({
    train: Dataset({
        features: ['filename', 'captions', 'image'],
        num_rows: 8734
    })
    test: Dataset({
        features: ['filename', 'captions', 'image'],
        num_rows: 1093
    })
    valid: Dataset({
        features: ['filename', 'captions', 'image'],
        num_rows: 1094
    })
})

In [3]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [4]:
import torch
from transformers import AutoModel
model = AutoModel.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True  # Required for Qwen models
)

Loading checkpoint shards: 100%|██████████| 5/5 [00:22<00:00,  4.59s/it]


In [5]:
from peft import PromptTuningConfig, get_peft_model,TaskType,PromptTuningInit

# Soft Prompt
config = PromptTuningConfig(
    task_type= TaskType.CAUSAL_LM,
    num_virtual_tokens = 4
)

In [6]:
model = get_peft_model(model, config)

In [7]:
model.print_trainable_parameters()


trainable params: 14,336 || all params: 7,070,633,472 || trainable%: 0.0002


In [16]:
args = TrainingArguments(
    output_dir="./prompttuning",
    per_device_train_batch_size=1,
    num_train_epochs=3,
    gradient_accumulation_steps=8,
    logging_steps=10,
    save_steps=100,
    save_total_limit=1,
    fp16=True,
    report_to="none"
)

In [17]:
MAX_LENGTH = 128

def preprocess(example):
    image = example["image"]
    prompt = "Describe the satellite image."
    encoding = processor(text=prompt, images=image, return_tensors="pt", padding="max_length", truncation=True, max_length=MAX_LENGTH)
    labels = processor.tokenizer(example["captions"][0], padding="max_length", truncation=True, max_length=MAX_LENGTH, return_tensors="pt").input_ids
    encoding["labels"] = labels.squeeze()
    return encoding

ds = ds.map(preprocess)
ds.set_format(type="torch")

Map:  91%|█████████▏| 999/1093 [00:23<00:02, 42.20 examples/s]


KeyboardInterrupt: 

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True)
)

In [27]:
import evaluate

In [28]:
cider = evaluate.load("cider")
spice = evaluate.load("spice")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = torch.argmax(torch.tensor(logits), dim=-1)
    decoded_preds = processor.tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = processor.tokenizer.batch_decode(labels, skip_special_tokens=True)
    return {
        "CIDEr": cider.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])["score"],
        "SPICE": spice.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])["score"]
    }

TypeError: 'NoneType' object is not callable

In [29]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds,
    eval_dataset=ds,
    tokenizer=processor.tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

/tmp/ipykernel_2619750/3122797283.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


KeyError: "Invalid key: 0. Please first select a split. For example: `my_dataset_dictionary['train'][0]`. Available splits: ['test', 'train', 'valid']"

In [ ]:
torch.cuda.reset_peak_memory_stats()
start = time.time()
sample = dataset[0]
out = model.generate(**processor(text="Describe this image.", images=sample["image"], return_tensors="pt").to(device))
end = time.time()
print("Generated:", processor.tokenizer.decode(out[0], skip_special_tokens=True))
print("Inference Time:", end - start, "seconds")
print("Peak VRAM:", torch.cuda.max_memory_allocated() / 1e9, "GB")